In [1]:
from pathlib import Path
parent_directory = Path.cwd().parent.parent
type(parent_directory)
print(parent_directory) 
type(parent_directory)

c:\Users\x286384\MerckGroup\Digitalización - Documents\DigitalProjectsLibrary\Planning


pathlib.WindowsPath

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
#reemplazar con la ruta de los archivos correcta
parent_directory = Path.cwd().parent.parent
FCST=pd.read_excel(parent_directory / "FCST Merck Abril 26.xlsx",sheet_name="Mayo 2026")
PROD_DC_2=pd.read_excel(parent_directory / "PRODUCTOS DC.xlsx",sheet_name="Catalogo")

In [3]:
"""""
Module: P&G Forecast extraction 2
Purpose: Extraer ## de jeringas para todos los meses por producto
Date: 30/07/2026
Author: J.Gonzalez
"""
id_cols = ['SKUMERCK','Description']

#Calculo de mes y año actual
Month_today=pd.to_datetime("today").month
Year_today=pd.to_datetime("today").year
demand_today=pd.to_datetime(str(Year_today) + "-01-" + str(Month_today), format="%Y-%m-%d")
demand_today=str(demand_today)[0:10]

#seleccion de meses con demanda futura en el FCST
matching_cols = []
for i in range(len(FCST.columns) - 1, -1, -1):  # from [-1] backwards
    col = FCST.columns[i]
    parsed = pd.to_datetime(col, errors='coerce')
    if pd.notna(parsed):
        matching_cols.append(col)
        if parsed.month <= Month_today and parsed.year <= Year_today:
            break  #
matching_cols = list(reversed(matching_cols))
hoy = pd.Timestamp.today()
seleccion = []
'''
for columna in reversed(list(FCST.columns)):
    fecha = pd.to_datetime(columna, errors="coerce")
    if pd.isna(fecha):
        continue
    seleccion.append(columna)
    if (fecha.year, fecha.month) <= (hoy.year, hoy.month):
        break
seleccion.reverse()
'''
#Union de columnas id con columna demanda de cada mes
FCST_month = FCST[id_cols + matching_cols].copy()
#FCST_month = FCST[id_cols + seleccion].copy()
FCST_month = FCST_month.dropna(subset=['SKUMERCK']).reset_index(drop=True)


In [ ]:
"""""
Module: Produccion por linea 2
Purpose: Separa los productos por familia, y despues por linea que utiliza dentro de cada mes esa familia especifica
Date: 30/07/2026
Author: J.Gonzalez
"""
#Merge de datos y eliminacion de SKU
PROD_DC_2['SKUMERCK'] = PROD_DC_2['SKUMERCK'].astype(str).str.strip()
FCST_month['SKUMERCK'] = FCST_month['SKUMERCK'].astype(str).str.strip()
PROD_DC_2 = PROD_DC_2.merge(FCST_month, on='SKUMERCK', how='left')
PROD_DC_2.columns = [pd.to_datetime(col, errors='coerce').strftime('%m-%Y') if pd.notna(pd.to_datetime(col, errors='coerce')) else col for col in PROD_DC_2.columns]
pre_grouping=PROD_DC_2.copy()
acondi=PROD_DC_2.copy()

PROD_DC_2.drop(id_cols, axis=1, inplace=True)

#dicccionario para la suma del group
agg_dict = {col: 'sum' for col in PROD_DC_2.columns if col not in ['Nombre granel', 'Linea Granel 1', 'Linea Granel 2']}

#Group by de familias de granel
PROD_DC_2 = PROD_DC_2.groupby(['Nombre granel', 'Linea Granel 1', 'Linea Granel 2'], as_index=False).agg(agg_dict)

#Se reduce de dos columnas a una para identificar DC
PROD_DC_2['Linea Granel 1'] = np.where(
    (PROD_DC_2['Linea Granel 1'] == 1) & (PROD_DC_2['Linea Granel 2'] == 1),
    'DC2',
    'DC1')
PROD_DC_2.drop(['Linea Granel 2'], axis=1, inplace=True)
PROD_DC_2 = PROD_DC_2.rename(columns={'Linea Granel 1': 'DC'})




In [5]:
'''
Module: Balanceo mensual 2(por producto)
Purpose: Balancea cada familia y mueve las cantidades de cada producto qu ese pueda fabricar en DC2 
Date: 24/08/2026
Author: J.Gonzalez
'''
logbalanceo=[]
# paso 1 id por LINEA POR FAMILIA POR NOMBRE DEL PRODUCTO
#demanda por familia
# Crea una copia de la fila del producto en dc2 sin demanda todavia. 

# Filtra las filas donde Linea Granel 2 es True
new_rows = pre_grouping[pre_grouping['Linea Granel 2'] == True][['Nombre granel', 'Linea Granel 1', 'Linea Granel 2', 'SKUMERCK','Description'] + list(agg_dict.keys())].copy()

# Establece los valores requeridos en las nuevas filas
pre_grouping.loc[new_rows.index, list(agg_dict.keys())] = 0
new_rows['Linea Granel 1'] = False

# Agrega las nuevas filas al DataFrame original
pre_grouping = pd.concat([pre_grouping, new_rows], ignore_index=True)
#Se reduce de dos columnas a una para identificar DC
pre_grouping['Linea Granel 1'] = np.where(
    (pre_grouping['Linea Granel 1'] == 0) & (pre_grouping['Linea Granel 2'] == 1),
    'DC2',
    'DC1')
pre_grouping.drop(['Linea Granel 2'], axis=1, inplace=True)
pre_grouping = pre_grouping.rename(columns={'Linea Granel 1': 'DC'})




In [6]:
"""""
Module: Balanceo mensual
Purpose: Balancea DC2 mandando su residuo a DC1 para cada familia en cada mes 
Date: 30/07/2026
Author: J.Gonzalez
"""
# Numero de jeringas por lote y redondeo de residuo
jeringas=115200
# Identifica todas las familias disponibles y registra cuales tienen capacidad en DC2
familias= PROD_DC_2['Nombre granel'].unique()
familias_con_dc2=PROD_DC_2.loc[PROD_DC_2['DC'] == 'DC2', 'Nombre granel'].unique()
# Identifica los productos con capacidad en DC2
prod_con_dc2 = pre_grouping.loc[pre_grouping['DC'] == 'DC2', 'SKUMERCK'].unique()
#Base de datos para residuos en jeringas
log_balanceo = pd.DataFrame(columns=['SKUMERCK', 'Description', 'Nombre granel', 'DC', 'fecha', 'log'] + list(agg_dict.keys()))

# Balanceo, en casos que detecte que la familia con DC2 tiene residuo mando todo ese residuo a DC1 de esa misma familia                      
for familia in familias_con_dc2:
    for col in agg_dict.keys():
        # Filtra las filas para esta familia específica a DC2 y DC1
        mask_dc2 = (PROD_DC_2['Nombre granel'] == familia) & (PROD_DC_2['DC'] == 'DC2')
        mask_dc1 = (PROD_DC_2['Nombre granel'] == familia) & (PROD_DC_2['DC'] == 'DC1')
        # Valor de DC2 para la familia y mes especificado
        dc2_val = PROD_DC_2.loc[mask_dc2, col].values[0]
        # Balanceo de residuos DC2
        if dc2_val % jeringas != 0:
            residue = dc2_val % jeringas
            # suma el residuo a DC1
            PROD_DC_2.loc[mask_dc1, col] = (PROD_DC_2.loc[mask_dc1, col] + residue)
            # resta el residuo de DC2
            PROD_DC_2.loc[mask_dc2, col] =(PROD_DC_2.loc[mask_dc2, col] - residue)

for prod in prod_con_dc2:
    for col in agg_dict.keys():
        mask_dc2 = (pre_grouping['SKUMERCK'] == prod) & (pre_grouping['DC'] == 'DC2')
        sku_dc2 = pre_grouping.loc[mask_dc2, col].values[0]
        if sku_dc2 % jeringas != 0:
            residue = sku_dc2 % jeringas      
            #Balanceo por producto en una familia:
            ##  busca el producto que pueda cumplir el minimo de jeringas de residuo para pasarlo 
            #   a dc1 y balancear dc2. Utiliza el residuo de un dc2 maximizado por productos y no
            #   por lineas   
            candidates = pre_grouping[
                (pre_grouping[col] > residue) & 
                (pre_grouping['SKUMERCK'] == prod) &
                (pre_grouping['DC'] == 'DC2')
            ][col]
            if candidates.empty:
                continue
            min_idx = candidates.idxmin()
            # Se resta el residuo de la fila para balancear DC2
            before_dc2 = pre_grouping.loc[[min_idx]] 
            pre_grouping.loc[min_idx, col] = pre_grouping.loc[min_idx, col] - residue    
            after_dc2 = pre_grouping.loc[[min_idx]].copy()           
            # Find the same SKU but with DC1
            dc1_mask = (pre_grouping['SKUMERCK'] == prod) & (pre_grouping['DC'] == 'DC1')
            before_dc1 = pre_grouping.loc[dc1_mask].copy()
            pre_grouping.loc[dc1_mask, col] += residue
            after_dc1 = pre_grouping.loc[dc1_mask].copy()

            for df, label in zip([before_dc2, before_dc1, after_dc2, after_dc1],
                                 ['before_dc2', 'before_dc1', 'after_dc2', 'after_dc1']):
                df['log'] = label
                df['fecha'] = col
            # Append all to log
            log_balanceo = pd.concat([log_balanceo, before_dc2, before_dc1, after_dc2, after_dc1], ignore_index=True)

PROD_DC_2[list(agg_dict.keys())] = PROD_DC_2[list(agg_dict.keys())] / jeringas

# Exportación de resultados en archivos CSV
DC1=PROD_DC_2[PROD_DC_2['DC'] == 'DC1'].copy()
DC2=PROD_DC_2[PROD_DC_2['DC'] == 'DC2'].copy()
DC1.to_csv("DC1_FCST.csv", index=False)
DC2.to_csv("DC2_FCST.csv", index=False)
PROD_DC_2.to_csv("PROD_DC_2_balanceado.csv", index=False)
log_balanceo.to_csv("log_balanceo.csv", index=False)
PROD_DC_2
print(sum(PROD_DC_2['09-2026'])/len(PROD_DC_2['09-2026']))



1.7810890151515153


In [7]:
"""""
Module: Capacidad de tiempos y campañas. 
Purpose: calcula las capacidades de tiempo por linea y susu campañas de 3 para un tercer balanceo.  
Date: 30/07/2026
Author: J.Gonzalez
"""
#115,200 jeringas es 27 horas por lote, en dc1 y dc2.
# campaña del mismo producto necesita ser maxima de 3 en 3 lotes. 
# los productos mas fuertes se deben fabricar despues. 
# se puede fabricar un lote individual pero despues de eso en automatico es limpieza
# cambio de familia afuerzas es limpieza
# 24/5 se programa el paro programado del fin de semana
        # nota: se debe calcular la cantidad de dias disponibles para 24/5 
        # es decir cuantos dias caen en sabado y cuantos caen en domingo
        # investigar función para calcular días laborales o desarollarla. 
#paso 1:dividir nuestros productos por dc1 y dc2 utilzando db: pre_grouping

pre_grouping.to_csv("pre_grouping.csv", index=False)
plan_prod_dc1 = pre_grouping[pre_grouping['DC'] == 'DC1'][['SKUMERCK', 'Nombre granel', pre_grouping.columns[3]]].copy()
plan_prod_dc2 = pre_grouping[pre_grouping['DC'] == 'DC2'][['SKUMERCK', 'Nombre granel', pre_grouping.columns[3]]].copy()
plan_prod_dc1 = plan_prod_dc1[plan_prod_dc1.iloc[:, -1] != 0]
plan_prod_dc2 = plan_prod_dc2[plan_prod_dc2.iloc[:, -1] != 0]

In [ ]:
prod_grouped = pre_grouping.copy()
prod_grouped = prod_grouped.drop(id_cols, axis=1)  # remove inplace=True
prod_grouped = prod_grouped.groupby(['Nombre granel', 'DC'], as_index=False).agg(agg_dict)
prod_grouped[list(agg_dict.keys())] = prod_grouped[list(agg_dict.keys())] / jeringas
prod_grouped
print(sum(PROD_DC_2['09-2026'])/len(PROD_DC_2['09-2026']))
prod_grouped

1.7810890151515153
